# K² FRS K² — Kaggle Model Lab (GPU T4x2)

**Institution**: Kakatiya University College of Engineering & Technology (KUCET), Warangal  
**Target Environment**: Kaggle Notebooks with **GPU T4 x 2** Accelerator  
**Models**: OpenCV Zoo YuNet (`face_detection_yunet_2023mar.onnx`) & SFace (`face_recognition_sface_2021dec.onnx`)  

---  
### Pipeline Overview
1. **Stage 00**: System & Hardware Audit (CPUs, RAM, Dual Tesla T4 GPUs, CUDA, OpenCV, ONNX Runtime).
2. **Stage 01**: Model Download & Cryptographic SHA-256 Integrity Verification.
3. **Stage 02**: YuNet Face Detection, Landmark Output & Zero-Detection Blank Image Verification.
4. **Stage 03**: SFace 5-Landmark Mathematical Alignment (`alignCrop`) & 128-D Embedding on $S^{127}$.
5. **Stage 04**: Exact Roster 1:N Retrieval, Margin Separation & Unknown Identity Rejection.
6. **Stage 05**: Multi-Batch Latency & Throughput Benchmark.
7. **Stage 06**: Provenance Telemetry & Manifest Export.


## 0. Install Kaggle Lab Dependencies

In [ ]:
!pip install -q -r requirements-kaggle.txt

## 1. Stage 00: Environment & Hardware Capabilities Audit

In [ ]:
import importlib
stage00 = importlib.import_module("scripts.00_environment_audit")
env_telemetry = stage00.run_audit()

## 2. Stage 01: Model Download & SHA-256 Hash Verification

In [ ]:
from pathlib import Path
stage01 = importlib.import_module("scripts.01_download_verify_models")

models_dir = Path("./models")
manifest_path = Path("./config/model_manifest.json")
verified_models = stage01.download_and_verify(models_dir, manifest_path)

## 3. Stage 02: YuNet Face Detection & Landmark Verification

In [ ]:
stage02 = importlib.import_module("scripts.02_face_detection_test")

yunet_model = models_dir / "face_detection_yunet_2023mar.onnx"
detection_results = stage02.run_detection_tests(yunet_model)

## 4. Stage 03: SFace 5-Landmark Alignment & 128-D Manifold Verification

In [ ]:
stage03 = importlib.import_module("scripts.03_face_recognition_test")

sface_model = models_dir / "face_recognition_sface_2021dec.onnx"
recognition_results = stage03.run_recognition_tests(sface_model)

## 5. Stage 04: Exact Roster-Scoped Retrieval & Threshold Calibration

In [ ]:
stage04 = importlib.import_module("scripts.04_retrieval_and_calibration")

retrieval_results = stage04.run_retrieval_experiment()

## 6. Stage 05: Batch Benchmark (Latency & Throughput)

In [ ]:
import matplotlib.pyplot as plt
stage05 = importlib.import_module("scripts.05_batch_benchmark")

bench_results = stage05.run_benchmark(sface_model, batch_sizes=[1, 4, 8, 16, 32], iterations=15)

# Plot Throughput vs Batch Size
batches = [res["batch_size"] for res in bench_results.values()]
throughputs = [res["throughput_fps"] for res in bench_results.values()]
latencies = [res["p50_latency_ms"] for res in bench_results.values()]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(batches, throughputs, "o-", color="#1E3A8A", label="Throughput (FPS)")
ax1.set_xlabel("Batch Size")
ax1.set_ylabel("Throughput (Images / sec)", color="#1E3A8A")
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(batches, latencies, "s--", color="#DC2626", label="p50 Latency (ms)")
ax2.set_ylabel("Latency (ms)", color="#DC2626")

plt.title("K² FRS SFace Inference Scaling (Kaggle Environment)")
plt.show()

## 7. Stage 06: Export Manifest & Run Summary

In [ ]:
stage06 = importlib.import_module("scripts.06_export_model_manifest")

output_file = Path("./output/kaggle_run_summary.json")
final_summary = stage06.export_summary(
    output_file,
    env_telemetry,
    verified_models,
    retrieval_results,
    bench_results,
)
print("Telemetry successfully exported to:", output_file)